In [ ]:
import pandas as pd
import os as os
import re
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')


pd.set_option('display.max_colwidth',700)
pd.set_option('display.max_columns',None)


In [ ]:

path = '../data/raw/WhatsApp Chat with Competitive Programmers.txt'


with open(path, encoding='utf-8') as file:
  file_data = file.read()


len(file_data)

In [ ]:
file_data

In [ ]:
file_data = file_data.replace('\u202f',' ')


In [ ]:

pattern = r'\d{1,2}/\d{1,2}/\d{2,4},\s\d{1,2}:\d{1,2}\s[AP]M\s-\s'

message = re.split(pattern,file_data)[1:]

message[1:20]


In [ ]:
dates = re.findall(pattern,file_data)

dates[1:10]

In [ ]:
len(message), len(dates)

In [ ]:
data = pd.DataFrame({'user_message':message,'message_dates':dates})

data.head()

In [ ]:
data.info()

In [ ]:
data['message_dates'] = (data['message_dates'].str.replace(' -',' ',regex=False).str.strip())


In [ ]:
data['message_dates'].iloc[0]

In [ ]:

data['message_dates'] = pd.to_datetime(data['message_dates'],format= '%m/%d/%y, %I:%M %p', errors='coerce')
# data.rename(columns={'message_dates': 'date'},inplace=True)

In [ ]:
data.info()

In [ ]:
data.head(10)

In [ ]:
data['user_message'].iloc[100:120]

In [ ]:

users = []
messages = []

for message in data['user_message']:
  # entry = re.split(r'([\w\W+]):\s',message)
  entry = re.split(r'(.*?):\s',message)
  if entry[1:]:
    users.append(entry[1])
    messages.append("".join(entry[2]))
  else:
    users.append('user-notification')
    messages.append(entry[0])


data['user'] = users
data['message'] = messages
data.drop(columns=['user_message'],inplace=True)


In [ ]:
 user_list = data['user'].unique().tolist()
  # user_list.remove('group_notification')
 user_list

In [ ]:
data.head()

In [ ]:
data['user'].nunique()

In [ ]:
data.dtypes

In [ ]:
data['message_dates'].dtype

In [ ]:
data['date'] = data['message_dates'].dt.date
data['day'] = data['message_dates'].dt.day
data['day_name'] = data['message_dates'].dt.day_name()
data['hour'] = data['message_dates'].dt.hour
data['minutes'] = data['message_dates'].dt.minute
data['month'] = data['message_dates'].dt.month
data['month_name'] = data['message_dates'].dt.month_name()
data['year'] = data['message_dates'].dt.year




In [ ]:
data.dtypes

In [ ]:
data.drop(columns=['message_dates'],inplace=True)

In [ ]:
period = []
for hour in data['hour']:
    if hour == 23:
        period.append(str(hour) + "-" + str('00'))
    elif hour == 0:
        period.append(str('00') + "-" + str(hour + 1))
    else:
        period.append(str(hour) + "-" + str(hour + 1))

data['period'] = period

data.head()

In [ ]:
# Display Basic Statistics for Data Analysis

data.shape

In [ ]:
data['message']

In [ ]:
words = data['message'].dropna().astype(str).str.replace('\n','').str.split().sum()

type(words)


In [ ]:
# Media files

data[data['message'] == '<Media omitted>\n'].shape[0]


In [ ]:
! pip install urlextract

In [ ]:
from urlextract import URLExtract

extract = URLExtract()

links = []

for link in data['message']:
  links.extend(extract.find_urls(link))

print(links)





In [ ]:
data.head()

In [ ]:

user = data['user'].value_counts().head()

user_name = user.index
count = user.to_numpy()

sns.set_theme(style='darkgrid', palette='viridis')

plt.figure(1, figsize=(10,6))
plt.bar(user_name,count)
# plt.xticks(rotation='vertical')


In [ ]:
# percentage

active_percent = round(((data['user'].value_counts() / data.shape[0]) * 100),2).reset_index().rename(columns={'index':'name','count':'percent'})

active_percent.head()

In [ ]:
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')
stopwords_list = set(stopwords.words('english'))

def hindi_stopwords(message):
  with open('/content/drive/MyDrive/Colab Notebooks/Supervised Learning/WhatsApp Chat Analysis/hindi_stop_words.txt', 'r') as file:
    stop_words = file.read()

  filtered_message = []
  for word in message.lower().split():
      if word not in stop_words:
          filtered_message.append(word)
  return " ".join(filtered_message)

def english_stopwrods(message):
  filtered_message = []
  for word in message.lower().split():
    if word not in stopwords_list:
      filtered_message.append(word)
  return " ".join(filtered_message)

def remove_punctuation(message):
  return  re.sub(r'[^\w\s]', '', message)



In [ ]:

temp = data[data['user'] != 'user-notification']
temp = temp[temp['message'] != '<Media omitted>\n']


temp['message'] = temp['message'].apply(hindi_stopwords)
temp['message'] = temp['message'].apply(english_stopwrods)
temp['message'] = temp['message'].apply(remove_punctuation)




In [ ]:
from wordcloud import WordCloud


wc = WordCloud(width= 1000, height= 750, min_font_size=20, random_state= 21, max_font_size=100).generate(temp['message'].str.cat(sep=' '))
plt.axis('off')
plt.imshow(wc)

In [ ]:
# Top 20 Most Common Words
from collections import Counter

words = []

for msg in temp['message']:
  words.extend(msg.split())



common_words = pd.DataFrame(Counter(words).most_common(20))

common_words



In [ ]:
! pip install emoji

In [ ]:
data['message'].iloc[100:120]

In [ ]:
import emoji

emoji_list = []

for msg in data['message']:
  emoji_list.extend([ c for c in msg if c in emoji.EMOJI_DATA])

common_emoji = pd.DataFrame(Counter(emoji_list).most_common(len(Counter(emoji_list))))


# common_emoji

#
plt.pie(common_emoji[1].head(10), labels=common_emoji[0].head(10), autopct="%0.2f%%")


In [ ]:
data.columns

In [ ]:
# Time-based Analysis

timeline = data.groupby(['month','month_name','year'])['message'].count().reset_index()

timeline.head()



In [ ]:
month_timeline = []

for i in range(timeline.shape[0]):
  month_timeline.append(timeline['month_name'][i]+ ' - '+ str(timeline['year'][i]))

timeline['monthly_timeline'] = month_timeline



plt.figure(figsize=(12,6))
plt.plot(timeline['monthly_timeline'], timeline['message'], alpha=0.8,  linestyle = '-.', marker = '*')
plt.xticks(rotation='vertical', fontsize=10)
plt.tight_layout()
plt.show()


In [ ]:
# Daily Timeline

daily = data.groupby('date')['message'].count().reset_index()

# timeline['daily_timeline'] = daily

sns.set_style('darkgrid')
plt.figure(figsize=(12,6))
plt.plot(daily['date'], daily['message'], color='green')
plt.xticks(rotation='vertical')
plt.show()



In [ ]:
# Day-based Activity Map

sns.set_style('darkgrid')
plt.title("Busy Day")
day_based = data['day_name'].value_counts().plot.bar(color='purple', alpha=0.7)




In [ ]:
data.columns

In [ ]:
# Monthly Activity Map



plt.title("Monthly Activity Map")
day_based = data['month_name'].value_counts().plot.bar(color='orange')
plt.xticks(rotation=45)


In [ ]:

plt.figure(1, figsize=(16,6))
sns.heatmap(data.pivot_table(index='day_name', columns='period', values='message',
            aggfunc='count').fillna(0))
plt.yticks(rotation=360)
plt.xticks(rotation=45)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import preprocess as process
import helper as helper
import streamlit as st


uploaded_file = st.sidebar.file_uploader('choose a file')

if uploaded_file is not None:
  byte_data = uploaded_file.get_value()
  data = byte_data.decode('utf-8')

  data = process.preprocess(data)

  user_list = data.unique().to_list()
  user_list.remove('group_notification')
  user_list.sort()
  user_list.insert(0,'Over all')

  selected_user = st.sidebar.selectbox("Show Analysis", user_list)

  if(st.sidebar.button == 'Show Analysis'):
     num_messages, words, num_media_messages, num_links = helper.fetch_stats(selected_user,data)
     col1, col2, col3, col4 = st.coloumns(4)

     with col1:
      st.header('Total Message')
      st.title(num_messages)
    with col2:
      st.header('Total Words')
      st.title(words)
    with col3:
      st.header('Media shared')
      st.title(num_media_messages)
    with col4:
      st.header('Links shared')
      st.title(num_links)


    # Monthly timeline

    st.title('Monthly Timeline')
    timeline = helper.monthly_timeline(selected_user, data)
    fig, ax = plt.subplots()
    ax.plot(timeline['monthly_timeline'], timeline['message'],color='cyan')
    ax.xtrick(rotation='vertical')
    st.pyplot(fig)

    # Daily timeline

    st.title('Daily Timeline')
    timeline = helper.daily_timeline(selected_user,data)
    fig, ax = plt.subplots()
    ax.plot(timeline['date'], timeline['message'],color='cyan')
    ax.xtrick(rotation='vertical')
    st.pyplot(fig)

     # Day-based Activity Map

     st.tile('Day-based Activity Map')
     col1, col2 = st.columns(2)

    with col1:
      st.title("Busy day")
      busy_day = helper.day_based_activity( selected_user,data)
      fig, ax = plt.subplots()
      ax.bar(busy_day.index,busy_day.to_numpy(),color='brown')
      ax.xtrick(totation='vertical')
      st.pyplot(fig)

    with col2:
      st.title("Busy month")
      busy_month = helper.month_based_activity(selected_user, data)
      fig, ax = plt.subplots()
      ax.bar(busy_month.index, busy_moth.to_numpy(), color='red')
      ax.xtrick(totation='vertical')
      st.pyplot(fig)

    # activity_heatmap
    st.title("Weekly Activity Map")
    fig, ax = plt.subplots()
    activity_heatmap = helper.activity_heatmap(selected_user, data)
    ax = plt.heatmap(activity_heatmap)
    ax.xtrick(totation='vertical')
    st.pyplot(fig)

    # Most most_busy_users

    st.title("Most most_busy_users")
    most_busy, stats = helper.most_busy_users(selected_user, data)
    fig, ax = plt.subplots()

    col1, col2 = st.columns(2)

    with col1:
      st.title("Most busy User")
      ax = plt.bar(most_busy.index, most_busy.to_numpy(), color='green')
      ax.xtrick(totation='vertical')
      st.pyplot(fig)


    with col1:
      st.title("Most busy User & stats")
      ax = plt.bar(stats.index, stats.user, color='blue')
      ax.xtrick(totation='vertical')
      st.pyplot(fig)

  # Word Cloud

  st.titel("Word Cloud")
  fig, ax = plt.subplots()
  wc = helper.wordcloud(selected_user, data)
  ax.imshow(wc)
  st.pyplot(fig)


  # most_common_words

  st.title('most_common_words')
  fig, ax = plt.subplots()
  most_common_words = helper.most_common_words(selected_user, data)
  ax.barh(most_common_words['common_words'], common_words['count'])
  ax.xtrick(totation='vertical')
  st.pyplot(fig)


  # emoji_helper

  st.title("Emoji Analysis")
  fig, ax = plt.subplots()
  emoji_helper = helper.emoji_helper(selected_user, data)
  col1, col2 = st.columns(2)

  if emoji_helper.shape[0] > 0:
    with col1:
      st.dataframe(emoji_helper)
    with col2:
      ax.pie(emoji_helper[1].head(), labels=emoji_helper[0].head(), autopct="%0.2f" )
      st.pyplot(fig)

  else:
     st.write(":red[No Emojis Send by this user]")







